# SumExplicitStack (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_java.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `SumExplicitStack`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [2]:
%%writefile ExplicitStack.java
import java.util.*;

public class ExplicitStack {
    static int maxP = 0;   // massima dimensione della pila P

    // sumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita
    static long sumExplicitStack(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();   // pila dei sotto-array da sommare
        P.push(new int[]{0, n - 1});
        maxP = Math.max(maxP, P.size());
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];           // caso base
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                maxP = Math.max(maxP, P.size());
            }
        }
        return s;
    }

    // la cima della pila è a destra: si stampa dal fondo alla cima
    static String fmtP(ArrayDeque<int[]> P) {
        StringBuilder sb = new StringBuilder("[");
        Iterator<int[]> it = P.descendingIterator();
        boolean first = true;
        while (it.hasNext()) {
            int[] x = it.next();
            if (!first) sb.append(",");
            sb.append("(").append(x[0]).append(",").append(x[1]).append(")");
            first = false;
        }
        return sb.append("]").toString();
    }

    static long sumExplicitStackTrace(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();
        P.push(new int[]{0, n - 1});
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];
                System.out.println("pop (" + l + "," + r + "): caso base, s=" + s + " => P=" + fmtP(P));
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                System.out.println("pop (" + l + "," + r + "): mid=" + mid + ", push (" + (mid + 1) + "," + r + ") e (" + l + "," + mid + ") => P=" + fmtP(P) + ", s=" + s);
            }
        }
        return s;
    }

    // ---- le altre somme del codice del corso, con la profondità ----
    static int depth = 0;
    static int maxDepth = 0;

    static long sumIt(int[] a, int n) {
        long s = 0;
        for (int i = 0; i < n; i++) {
            s = s + a[i];
        }
        return s;
    }

    static long sumRecHalving(int[] a, int l, int r) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (l == r) {
            result = a[l];
        } else {
            int mid = l + (r - l) / 2;
            result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
        }
        depth--;
        return result;
    }

    static long sumRecDecrease(int[] a, int n) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (n == 0) {
            result = 0;
        } else {
            result = sumRecDecrease(a, n - 1) + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
    static int stepNo = 0;
    static int maxActive = 0;

    static void emit(ArrayList<String> stack, String action) {
        stepNo++;
        if (stack.size() > maxActive) maxActive = stack.size();
        System.out.printf("%2d | %s | %s%n", stepNo, String.join(" ", stack), action);
    }

    static long sumRecDecreaseTrace(int[] a, int n, ArrayList<String> stack) {
        stack.add("s(" + n + ")");
        if (n == 0) {
            emit(stack, "n=0: return 0");
            stack.remove(stack.size() - 1);
            return 0;
        }
        emit(stack, "n=" + n + " != 0: chiama s(" + (n - 1) + "), poi somma a[" + (n - 1) + "]");
        long r = sumRecDecreaseTrace(a, n - 1, stack);
        emit(stack, "return " + r + "+a[" + (n - 1) + "]=" + (r + a[n - 1]));
        stack.remove(stack.size() - 1);
        return r + a[n - 1];
    }

    public static void main(String[] args) {
        int[] a = {4, 7, 1, 9};
        int n = 4;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("SumExplicitStack(a, 4):");
        long res = sumExplicitStackTrace(a, n);
        maxP = 0;
        sumExplicitStack(a, n);
        System.out.println("risultato " + res + ", dimensione massima di P = " + maxP);
        System.out.println();
        System.out.println("sumIt                = " + sumIt(a, n));
        maxDepth = 0;
        long r1 = sumRecHalving(a, 0, n - 1);
        System.out.println("sumRecHalving        = " + r1 + "   profondità massima = " + maxDepth);
        maxDepth = 0;
        long r2 = sumRecDecrease(a, n);
        System.out.println("sumRecDecrease       = " + r2 + "   profondità massima = " + maxDepth);

        int[] b = {4, 7, 1};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   sumRecDecrease(a, 3)");
        System.out.println(" # | pila delle chiamate attive | azione");
        res = sumRecDecreaseTrace(b, 3, new ArrayList<>());
        System.out.println("risultato " + res + ", chiamate attive al massimo = " + maxActive + " = n+1");
    }
}

Writing ExplicitStack.java


In [3]:
!javac ExplicitStack.java && java ExplicitStack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [4]:
%%writefile ProvaExplicitStack.java
import java.util.*;

public class ProvaExplicitStack {
    public static void main(String[] args) {
        // Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
        int[] c = new int[100];
        for (int i = 0; i < 100; i++) c[i] = i + 1;
        ExplicitStack.maxP = 0;
        long s = ExplicitStack.sumExplicitStack(c, 100);
        System.out.println(s + " " + ExplicitStack.maxP);
    }
}

Writing ProvaExplicitStack.java


In [5]:
!javac ProvaExplicitStack.java && java ProvaExplicitStack

5050 8


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).